In [1]:
# 수학 체인
# 사용자 질문 >> 수학 함수 호출 체인

!pip install langchain
!pip install openai
!pip install langchain_openai

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 974.2/974.2 kB 6.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 315.5/315.5 kB 9.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 125.2/125.2 kB 1.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 145.0/145.0 kB 3.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 325.5/325.5 kB 4.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 75.6/75.6 kB 292.8 kB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 77.9/77.9 kB 4.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.3/58.3 kB 6.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 19.9 MB/s eta 0:00:00


In [2]:
import os
import openai

from google.colab import userdata
os.environ["OPENAI_API_KEY"] = userdata.get('OPENAI_API_KEY')

In [3]:
# 수학공식 푸는 패키지 numexpr

import numexpr

In [4]:
numexpr.evaluate('15**0.3432')

array(2.53299609)

In [5]:
numexpr.evaluate('15**2')

array(225, dtype=int32)

In [7]:
# 내가 원하는 거
# 자연어 질문 >> 수학공식 생성 >> numexpr >> 답변

from langchain.prompts import PromptTemplate
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI
from langchain.prompts.chat import ChatPromptTemplate, SystemMessagePromptTemplate, HumanMessagePromptTemplate, AIMessagePromptTemplate


# PromptTemplate : 대화의 전반적인 흐름 제어하는 템플릿
# SystemMessagePromptTemplate : 시스템 메시지만 포함하는 템플릿
# HumanMessagePromptTemplate: 사용자 메시지만 포함하는 템플릿

In [8]:
# numexpr.evaluate("수학공식")
# 자연어 >> llm >> 수학공식 >> parsing >> numexpr.evalute

# 출력형식
'''
```text
13**0.3432
```
'''
# ai 가 정해진 질문 이해, 답변 role (역할) 부여

system_content_template = """You are a helpful assistant that make an only ARGUMENT for numexpr.evaluate according to user query such as
```text
13**0.3432
```
when user query is 'What is 13 raised to the .3432 power?'

Please response as code block using text back quote like
```text
13**0.3432
```
without other description."""

In [9]:
# 실제 사용자가 어떤 형식으로 input 전달할지 정의
human_content_template = "{user_input}"

# 템플릿 정의
chat_prompt_template = ChatPromptTemplate.from_messages([
  ("system", system_content_template),
  ('human', human_content_template)
])

chat_model = ChatOpenAI(temperature=0)

chain = chat_prompt_template | chat_model

chain.invoke({"user_input":"What is 15 raised to the .3432 power?"})

AIMessage(content='```text\n15**0.3432\n```', response_metadata={'token_usage': {'completion_tokens': 11, 'prompt_tokens': 101, 'total_tokens': 112}, 'model_name': 'gpt-3.5-turbo', 'system_fingerprint': None, 'finish_reason': 'stop', 'logprobs': None}, id='run-4fe03c5e-9591-4a4b-9fcd-1ec13156911e-0', usage_metadata={'input_tokens': 101, 'output_tokens': 11, 'total_tokens': 112})

In [10]:
# 모델의 출력을 특정 형식으로 구문 분석 >> 필요에 따라 추가적인 처리를 수행하는 데 사용
from langchain.schema import BaseOutputParser

# ```text\n15**0.3432\n``` >> "15**0.3432"

class CodeOutputParser(BaseOutputParser):
  def parse(self, text: str):
    # text 변수 문자열(str)  전달되어야 함 명시
    value = text.strip().split("```")
    # 공백 제거, ``` 기준 구분 (분리)
    print(value)
    return value[1].split("\n")[1]


chain = chat_prompt_template | chat_model | CodeOutputParser()

output = chain.invoke({"user_input":"What is 15 raised to the .3432 power?"})
print(output)

['', 'text\n15**0.3432\n', '']
15**0.3432


In [13]:
chain = chat_prompt_template | chat_model | CodeOutputParser() | numexpr.evaluate

output = chain.invoke({"user_input":"What is 15 raised to the .3432 power?"})
print(output)

['', 'text\n15**0.3432\n', '']
2.5329960941138205


In [16]:
from langchain_core.runnables import RunnablePassthrough, RunnableParallel, RunnableLambda

chain = ({'user_input': RunnablePassthrough()}
        | chat_prompt_template
        | chat_model
        | CodeOutputParser()
        | numexpr.evaluate

         )

output = chain.invoke("What is 15 raised to the .3432 power?")
print(output)

['', 'text\n15**0.3432\n', '']
2.5329960941138205
